# Task 2 — Definição da previsão e da avaliação

**Mineração de Dados · ISEP · 2026/27 1.º semestre**

| Secção | O que responde | Tipo de entrega |
|---|---|---|
| 1.1 | Definição formal do problema | relatório |
| 1.2 | Variáveis nas 4 categorias do enunciado | relatório + tabela |
| 1.3 | Falsos positivos vs falsos negativos e critério do limiar | relatório |
| 1.4 | Métricas: principal, complementares, operacionais, estabilidade | relatório |
| 1.5 | Função `avaliar()` | notebook (+ `avaliacao.py`) |

**Convenções deste notebook**

- As decisões do grupo estão marcadas com **[DECISÃO]** e foram tomadas **antes de qualquer modelo ser treinado**.
- O **conjunto de teste (out–dez/2021) é removido logo na secção 0** e nunca é usado aqui.

| # | Decisão | Escolha | Onde está justificada |
|---|---|---|---|
| 1 | O alerta é sobre a sessão ou sobre o posto? | A previsão é feita **por sessão**; o alerta serve para **vigiar e manter o posto** | 1.1 |
| 2 | Orçamento de alertas **X** | **X = 200 alertas por 1000 sessões** (20 % das sessões) | 1.3 |
| 3 | Categoria do `Order creation time` | **Categoria 4** (excluída por outra razão metodológica) | 1.2 |
| 4 | Métrica principal | **Recall com 200 alertas por 1000 sessões** (revisto antes de treinar modelos) | 1.4, 2.3 |
| 5 | Limiar no teste | **Móvel**: recalculado à segunda-feira com os scores dos 7 dias anteriores | 2.4 |
| 6 | Agregação dos folds | **Média simples** + pior fold + comparação emparelhada fold a fold | 2.5 |

## 0. Preparação

**O que fazer:** carregar os dados, derivar o alvo e separar já o teste. Só são lidas as colunas necessárias para esta parte.

In [1]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import avaliacao
from avaliacao import avaliar, avaliar_por_mes, limiar_por_orcamento

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

DATA_PATH = Path("Charging_Data_educational.csv")
NORMAL_CAUSES = ["Charging ends normally", "User stops charging"]

# Fronteiras do split cronológico definido na Task 1
TRAIN_END = pd.Timestamp("2021-07-01")   # treino: dez/2019 – jun/2021
VAL_END   = pd.Timestamp("2021-10-01")   # validação: jul–set/2021 (teste: out–dez/2021)

COLUNAS_ORIGINAIS = pd.read_csv(DATA_PATH, nrows=0).columns.str.strip().tolist()

TEMPOS = ["Order creation time", "Start Time", "End Time", "Payment time"]
df = pd.read_csv(DATA_PATH, usecols=["UserID", "end_cause", *TEMPOS], parse_dates=TEMPOS)
df["end_cause"] = df["end_cause"].str.strip()
df["is_Abnormal"] = (~df["end_cause"].isin(NORMAL_CAUSES)).astype(int)

n_total = len(df)
# O teste deixa de existir neste notebook:
dev = df[df["Start Time"] < VAL_END].copy()
train = dev[dev["Start Time"] < TRAIN_END]
val = dev[dev["Start Time"] >= TRAIN_END]
del df

print(f"{len(COLUNAS_ORIGINAIS)} colunas no ficheiro | {n_total:,} sessões no total")
print(f"Desenvolvimento: {len(dev):,} (treino {len(train):,} + validação {len(val):,})")

import time
T_INICIO = time.perf_counter()      # para medir o tempo total do caderno

47 colunas no ficheiro | 441,077 sessões no total
Desenvolvimento: 360,996 (treino 285,227 + validação 75,769)


## 1.1 Definição do problema

**Resumo**

| Elemento | Definição |
|---|---|
| Unidade de observação | Uma sessão de carregamento (uma linha = uma transação de um utilizador num posto). |
| Classe positiva (`is_Abnormal = 1`) | Sessão anómala: `end_cause` diferente de "Charging ends normally" e "User stops charging" (13 causas de falha). |
| Classe negativa (`is_Abnormal = 0`) | Sessão normal: fim normal da carga ou paragem pelo próprio utilizador. A paragem voluntária não é uma falha do equipamento nem do veículo. |
| Derivação do alvo | `is_Abnormal = 1` se `end_cause` não pertencer às duas causas normais. A definição foi validada na Task 1: reconstruindo a flag user_no_previous_completed_abnormal a partir deste alvo, o resultado coincide em 100 % das sessões, contra 87,96 % se a paragem do utilizador fosse considerada anómala. |
| Momento da previsão | O instante `Start Time` da sessão. Só pode ser usada informação conhecida nesse instante (contexto, meteorologia prevista, tarifa e histórico de sessões já concluídas). |
| Finalidade | Monitorização preventiva, priorização de inspeção e manutenção, e melhoria da qualidade de serviço. |
| Significado de prever "anómala" | Um **alerta de risco elevado** de a sessão terminar em falha. Não é um diagnóstico da causa nem uma garantia de que a falha ocorre; serve para desencadear uma ação (ver 1.3). |

**[DECISÃO]** A previsão é feita ao nível da **sessão** (é a unidade de observação), mas o alerta destina-se sobretudo a apoiar a **vigilância e a manutenção do posto**. As falhas concentram-se em postos que já falharam (relatório da Task 1, secção 5.2), por isso a ação útil é agir sobre o posto e não sobre a sessão individual. Um alerta significa "este posto, neste momento, tem um risco elevado de falha".

In [2]:
# Números de apoio ao 1.1 (apenas treino e validação)
for nome, d in [("treino", train), ("validação", val)]:
    print(f"{nome:10s} n={len(d):7,}  prevalência de anómalas = {d['is_Abnormal'].mean():.1%}")

causas = (dev.groupby("end_cause")["is_Abnormal"]
             .agg(n="size", anomala="max")
             .sort_values("n", ascending=False))
causas["% das sessões"] = (causas["n"] / len(dev) * 100).round(2)
causas

treino     n=285,227  prevalência de anómalas = 18.2%
validação  n= 75,769  prevalência de anómalas = 21.1%


,n,anomala,% das sessões
end_cause,,,
Charging ends normally,165641,0,45.88
User stops charging,127509,0,35.32
Other faults of the charging pile,16917,1,4.69
DC Output Fault,12587,1,3.49
Vehicle communications failure,11620,1,3.22
Faulty charging connection,7014,1,1.94
BMS failure,6584,1,1.82
Failure of non-system control loop components and assemblies,6123,1,1.70
Vehicle charging faults,4768,1,1.32


## 1.2 Variáveis nas 4 categorias do enunciado

In [3]:
CAT = {1: "1. Preditor elegível",
       2: "2. Só para construir o alvo",
       3: "3. Gerada durante/depois da sessão",
       4: "4. Excluída por outra razão metodológica"}

regras = []   # (variável, categoria, justificação, é derivada?)
def add(nomes, cat, just, derivada=False):
    for n in ([nomes] if isinstance(nomes, str) else nomes):
        regras.append((n, cat, just, derivada))

# --- Categoria 2: só para construir o alvo
add("end_cause", 2, "Define o alvo e só é conhecida no fim da sessão (fuga de informação direta).")

# --- Categoria 3: geradas durante ou depois da sessão
add("End Time", 3, "Só existe quando a sessão termina.")
add("Payment time", 3, "Registado depois do fim da sessão.")
add("Transaction power/kwh", 3, "Energia acumulada durante a sessão; as sessões anómalas têm energia nula ou duração curta (consequência do fim, não causa).")
add(["Electricity cost/Yuan", "Service charge/Yuan", "Transaction Amount/Yuan", "Actual Payment/Yuan"],
    3, "Calculados a partir da energia consumida; só são conhecidos no fim da sessão.")

# --- Categoria 4: outra razão metodológica
add("Order creation time", 4, "Coincide com o início na quase totalidade das sessões (redundante com Start Time); "
    "num pequeno número (131 no desenvolvimento, 0,04 %) é registada perto do fim (fuga de informação) "
    "e o sentido da diferença está associado ao alvo. Não é fiável como informação de início.")
add("UserID", 4, "Identificador de cardinalidade muito alta; grande parte das sessões de validação é de contas não vistas no treino (não generaliza) e identifica pessoas (privacidade). O comportamento do utilizador entra pelos atributos user_*.")
add("tariff_period", 4, "Função determinística de `hour` (redundante).")
add("is_new_user", 4, "Redundante com user_no_previous_completed_session (ρ ≈ 0,99) e inflacionado no arranque dos dados.")
add(["user_previous_sessions_730D", "user_previous_abnormal_730D", "user_previous_abnormal_rate_730D"],
    4, "Janela de 730 dias nunca fica completa (dados com 2 anos) e é quase redundante com a de 180 dias (ρ ≥ 0,97).")
add(["post_no_previous_completed_session", "post_no_previous_completed_abnormal"],
    4, "Só valem 1 nas primeiras semanas (arranque, até mar/2020); em operação são constantes.")

# --- Categoria 1: tudo o resto é elegível (grupo para a ablação posterior)
atribuidas = {r[0] for r in regras}
GRUPOS = {
    "meteorologia": ["Temperature(℃)", "Relative Humidity(%)", "Precipitation(mm)"],
    "tarifa": ["electricity_price"],
    "localização": ["Charging Post ID", "Location Information", "District Name"],
}
def grupo(nome):
    for g, cols in GRUPOS.items():
        if nome in cols: return g
    if nome == "Start Time": return "tempo"
    if nome.startswith("post_"): return "histórico do posto"
    if nome.startswith("user_"): return "histórico do utilizador"
    return "?"

for c in COLUNAS_ORIGINAIS:
    if c not in atribuidas:
        just = ("Instante da previsão; não entra como tal, é a fonte de `hour` e `dow`." if c == "Start Time"
                else "Conhecida no início da sessão (ou histórico de sessões já concluídas).")
        add(c, 1, just)

# --- Variáveis derivadas (não existem no ficheiro original)
add(["hour", "dow"], 1, "Derivadas de Start Time (determinísticas, sem aprender dos dados).", derivada=True)
add(["post_last_completed_abnormal", "user_last_completed_abnormal"], 1,
    "Indicam se a última sessão concluída do posto/utilizador foi anómala (derivadas dos atributos históricos).", derivada=True)
add("is_weekend", 4, "Derivada de `dow` (redundante).", derivada=True)
add("month", 4, "Não usada: só 2 anos de dados, sem ciclo anual repetido suficiente para aprender sazonalidade.", derivada=True)

tabela = pd.DataFrame(regras, columns=["variável", "cat", "justificação", "derivada"])
tabela["categoria"] = tabela["cat"].map(CAT)
tabela["grupo"] = [grupo(v) if (c == 1 or v in ("hour", "dow")) else "" for v, c in zip(tabela["variável"], tabela["cat"])]
tabela.loc[tabela["variável"].isin(["hour", "dow"]), "grupo"] = "tempo"
tabela.loc[tabela["variável"].str.endswith("last_completed_abnormal"), "grupo"] = tabela["variável"].map(
    lambda v: "histórico do posto" if v.startswith("post_") else "histórico do utilizador")

# --- Verificações
originais = tabela[~tabela["derivada"]]["variável"]
assert set(originais) == set(COLUNAS_ORIGINAIS), set(COLUNAS_ORIGINAIS) ^ set(originais)
assert originais.is_unique
preditores = tabela[(tabela["cat"] == 1) & (tabela["variável"] != "Start Time")]
print(f"Colunas originais classificadas: {len(originais)}/{len(COLUNAS_ORIGINAIS)}")
print(f"Preditores elegíveis (sem Start Time): {len(preditores)}")
print(tabela.groupby("categoria")["variável"].count().to_string())

Colunas originais classificadas: 47/47
Preditores elegíveis (sem Start Time): 33
categoria
1. Preditor elegível                        34
2. Só para construir o alvo                  1
3. Gerada durante/depois da sessão           7
4. Excluída por outra razão metodológica    11


In [4]:
# Tabela completa por categoria (para o relatório: copiar as exclusões 2, 3 e 4)
tabela[tabela["cat"] != 1][["categoria", "variável", "justificação"]].sort_values(["categoria", "variável"]).reset_index(drop=True)

,categoria,variável,justificação
0,2. Só para construir o alvo,end_cause,Define o alvo e só é conhecida no fim da sessão (fuga de informação direta).
1,3. Gerada durante/depois da sessão,Actual Payment/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
2,3. Gerada durante/depois da sessão,Electricity cost/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
3,3. Gerada durante/depois da sessão,End Time,Só existe quando a sessão termina.
4,3. Gerada durante/depois da sessão,Payment time,Registado depois do fim da sessão.
5,3. Gerada durante/depois da sessão,Service charge/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
6,3. Gerada durante/depois da sessão,Transaction Amount/Yuan,Calculados a partir da energia consumida; só são conhecidos no fim da sessão.
7,3. Gerada durante/depois da sessão,Transaction power/kwh,"Energia acumulada durante a sessão; as sessões anómalas têm energia nula ou duração curta (consequência do fim, não causa)."
8,4. Excluída por outra razão metodológica,Order creation time,"Coincide com o início na quase totalidade das sessões (redundante com Start Time); num pequeno número (131 no desenvolvimento, 0,04 %) é registada perto do fim (fuga de informação) e o sentido da diferença está associado ao alvo. Não é fiável como informação de início."
9,4. Excluída por outra razão metodológica,UserID,Identificador de cardinalidade muito alta; grande parte das sessões de validação é de contas não vistas no treino (não generaliza) e identifica pessoas (privacidade). O comportamento do utilizador entra pelos atributos user_*.


In [5]:
# Preditores elegíveis por grupo (serão os grupos da ablação)
(preditores.groupby("grupo")["variável"].apply(list)
           .rename("variáveis").to_frame())

,variáveis
grupo,
histórico do posto,"[post_previous_sessions_1D, post_previous_abnormal_1D, post_previous_abnormal_rate_1D, post_previous_sessions_7D, post_previous_abnormal_7D, post_previous_abnormal_rate_7D, post_previous_sessions_30D, post_previous_abnormal_30D, post_previous_abnormal_rate_30D, post_hours_since_previous_completed_session, post_hours_since_previous_completed_abnormal, post_last_completed_abnormal]"
histórico do utilizador,"[user_previous_sessions_30D, user_previous_abnormal_30D, user_previous_abnormal_rate_30D, user_previous_sessions_180D, user_previous_abnormal_180D, user_previous_abnormal_rate_180D, user_days_since_previous_completed_session, user_no_previous_completed_session, user_days_since_previous_completed_abnormal, user_no_previous_completed_abnormal, user_active_sessions_at_start, user_last_completed_abnormal]"
localização,"[Charging Post ID, Location Information, District Name]"
meteorologia,"[Temperature(℃), Relative Humidity(%), Precipitation(mm)]"
tarifa,[electricity_price]
tempo,"[hour, dow]"


In [6]:
# ── Evidência 1: UserID — contas da validação que não existem no treino
vistos = set(train["UserID"])
nao_vistos = ~val["UserID"].isin(vistos)
print(f"Sessões de validação de contas não vistas no treino: {nao_vistos.mean():.1%}")

# ── Evidência 2: Order creation time — antes ou depois do início da sessão?
atraso_s = (dev["Order creation time"] - dev["Start Time"]).dt.total_seconds()      # >0: ordem criada DEPOIS do início
dist_fim_min = (dev["Order creation time"] - dev["End Time"]).dt.total_seconds().abs() / 60
tarde = atraso_s > 60 * 60
print(f"Mediana de (criação − início): {atraso_s.median():.0f} s")
print(f"Sessões com ordem criada mais de 60 min depois do início: {tarde.sum()} "
      f"({tarde.mean():.3%} do desenvolvimento)")
print(f"  dessas, registadas a ≤ 2 min do fim da sessão: {(dist_fim_min[tarde] <= 2).mean():.1%}")

Sessões de validação de contas não vistas no treino: 39.8%
Mediana de (criação − início): 8 s
Sessões com ordem criada mais de 60 min depois do início: 131 (0.036% do desenvolvimento)
  dessas, registadas a ≤ 2 min do fim da sessão: 73.3%


## 1.3 Falsos positivos vs falsos negativos

**Resumo**

| Erro | O que acontece | Consequência |
|---|---|---|
| **Falso negativo** (sessão anómala não detetada) | Nenhum alerta; a falha ocorre sem aviso. | Sessão interrompida e utilizador prejudicado; a avaria tende a repetir-se, pois a Task 1 mostra que, quando a sessão anterior do posto foi anómala, a probabilidade de a seguinte também o ser sobe para cerca de 50 % (46 % com intervalo longo, 60 % se a sessão começar logo a seguir), contra ~18 % de base; equipamento a falhar durante mais tempo sem intervenção. |
| **Falso positivo** (sessão normal sinalizada) | Alerta sem falha. | Inspeção ou intervenção desnecessária (custo de deslocação e de tempo), e **fadiga de alertas**: se muitos alertas forem falsos, a equipa deixa de os tratar. |

| Ação após o alerta | Custo | Tolera mais falsos positivos? |
|---|---|---|
| Monitorização automática reforçada do posto | Muito baixo | Sim |
| Priorização na fila de manutenção | Baixo a médio | Moderadamente |
| Inspeção técnica no local | Alto | Não |
| Intervenção junto do utilizador (aviso, sugestão de outro posto) | Médio (afeta a experiência) | Pouco |

**[DECISÃO]** O **falso negativo pesa mais**, porque as falhas são recorrentes e concentradas em postos que já falharam, pelo que deixar passar uma falha prolonga o problema. Mas o valor da deteção só se concretiza se a equipa conseguir tratar os alertas, o que impõe um **orçamento de alertas**. Daí o critério:

> **Maximizar o recall sujeito a um máximo de X alertas por 1000 sessões.**

Expressar o limiar como orçamento de alertas, e não como uma probabilidade fixa, permite adaptá-lo à variação
do risco entre períodos e à recalibração do modelo, **desde que o limiar seja recalculado periodicamente** com os
scores mais recentes. Um limiar fixado uma única vez não respeita o orçamento quando o nível de falhas muda (ver 2.4).

A célula seguinte mostra o que X significa em termos de recall alcançável, no treino e na validação.

In [7]:
# O que significa um orçamento de X alertas por 1000 sessões? (prevalência do treino)
prev = train["is_Abnormal"].mean()
prev_val = val["is_Abnormal"].mean()
X = np.array([50, 100, 150, 200, 300, 500])

meaning = pd.DataFrame({
    "alertas por 1000": X,
    "recall se o ranking fosse aleatório": X / 1000,
    "recall máx. possível (treino)": np.minimum(1, X / (prev * 1000)),
    "recall máx. possível (validação)": np.minimum(1, X / (prev_val * 1000)),
    "precisão máx. possível (treino)": np.minimum(1, prev * 1000 / X),
}).round(3)
print(f"Prevalência no treino: {prev:.1%} → ~{prev*1000:.0f} anómalas por 1000 sessões")
print(f"Prevalência na validação: {prev_val:.1%} → ~{prev_val*1000:.0f} anómalas por 1000 sessões")
meaning

Prevalência no treino: 18.2% → ~182 anómalas por 1000 sessões
Prevalência na validação: 21.1% → ~211 anómalas por 1000 sessões


,alertas por 1000,recall se o ranking fosse aleatório,recall máx. possível (treino),recall máx. possível (validação),precisão máx. possível (treino)
0,50,0.05,0.275,0.237,1.000
1,100,0.10,0.550,0.473,1.000
2,150,0.15,0.825,0.710,1.000
3,200,0.20,1.000,0.946,0.909
4,300,0.30,1.000,1.000,0.606
5,500,0.50,1.000,1.000,0.363


In [8]:
# >>> [DECISÃO] orçamento de alertas, fixado antes de treinar qualquer modelo <<<
X_ALERTAS_1000 = 200

**[DECISÃO] X = 200 alertas por 1000 sessões (20 % das sessões).**

1. Em cada 1000 sessões há cerca de 182 anómalas no treino e 211 na validação. X = 200 é da mesma ordem de grandeza do número de falhas esperadas; com X = 150, mesmo um modelo perfeito só detetaria cerca de 71 % das falhas na validação, o que seria demasiado restritivo.
2. A maioria das ações após um alerta é de custo baixo (monitorização reforçada, prioridade na fila de manutenção), o que tolera uma proporção elevada de falsos alertas.
3. As inspeções no local, que são caras, podem ser reservadas aos alertas com score mais alto dentro deste orçamento.

X foi fixado **antes de qualquer modelo ser treinado** e não será ajustado com base nos resultados. A sensibilidade a X ∈ {100, 150, 200, 300} é reportada na Task 3, na ablação.

## 1.4 Métricas


**Resumo**

| Métrica | Papel | O que mede | Porque é relevante | Limitação neste problema |
|---|---|---|---|---|
| **Recall com 200 alertas por 1000 sessões** | **Principal** | Fração das sessões anómalas que estão entre as 200 sessões por 1000 com score mais alto. | Mede diretamente o critério de decisão de 1.3; com o número de alertas fixo, mais recall implica também mais precisão. | Avalia um único ponto da curva e depende de X (a sensibilidade a 100, 150, 200 e 300 é reportada); quando há mais falhas do que alertas, o máximo possível é inferior a 1 (0,946 no F4). |
| **PR-AUC normalizada** = (PR-AUC − prevalência) ÷ (1 − prevalência) | Complementar | Qualidade da ordenação em todos os limiares: 0 = aleatório, 1 = perfeito. | Não depende do orçamento e é comparável entre folds com prevalências diferentes. | Inclui limiares que nunca serão usados; reduz, mas não elimina, a dependência da prevalência. |
| **PR-AUC** (precisão média) | Complementar | Compromisso entre precisão e recall ao longo de todos os limiares. | Centra-se na classe rara e não depende do limiar. | Depende da prevalência (18,2 % no treino e 21,1 % na validação); entre folds só é comparável na versão normalizada. |
| **Recall** | Decisão | Fração das anómalas que geram alerta. | Traduz o custo do falso negativo, o erro que mais pesa. | Trivial (100 %) se alertarmos tudo; só tem sentido sob orçamento de alertas. |
| **Precisão** | Decisão | Fração dos alertas que são de facto anómalas. | Traduz o custo dos falsos positivos e a fadiga de alertas. | Depende do limiar e da prevalência. |
| **F2** | Decisão | Média harmónica de precisão e recall, com o recall a pesar 2×. | Resume o compromisso favorecendo o recall, coerente com 1.3. | Depende do limiar; um só número esconde o orçamento de alertas. |
| **Alertas corretos / falsos alertas / falhadas / total por 1000 sessões** | Operacional | Carga de trabalho e falhas deixadas passar, por 1000 sessões. | Dá uma leitura prática: quantos alertas a equipa recebe e quantas falhas escapam. | Dependem do limiar e da prevalência do período; só são comparáveis com o mesmo orçamento. |
| **Desvio-padrão entre folds, pior fold, métrica por mês** | Estabilidade | Variação do desempenho ao longo do tempo. | Um modelo instável não deve ganhar só por ter melhor média (enunciado). | Poucos folds (4) tornam o desvio-padrão pouco preciso; meses com poucas sessões geram valores ruidosos. |

**Exatidão (accuracy): não é usada.** Com 79 % a 82 % de sessões normais (conforme o período), um classificador que diga sempre "normal" já tem essa exatidão sem detetar nenhuma falha.
**[DECISÃO] Revisão da métrica principal (antes de treinar qualquer modelo).** A primeira escolha foi a PR-AUC. No entanto, o critério de decisão de 1.3 é detetar o maior número de falhas com no máximo 200 alertas por 1000 sessões, e a PR-AUC avalia também limiares que nunca serão usados. As baselines da secção 2.3 mostram que isto muda a ordem dos métodos: a regra "a última sessão concluída do posto foi anómala" tem cerca de metade da PR-AUC normalizada da taxa do posto a 7 dias (0,201 contra 0,404), mas deteta mais falhas com 200 alertas em todos os folds (recall médio de 0,511 contra 0,499). A métrica principal passa, por isso, a ser o recall com 200 alertas por 1000 sessões, e a PR-AUC normalizada fica como complementar.

In [9]:
# Porque a exatidão não serve: classificador "sempre normal"
for nome, d in [("treino", train), ("validação", val)]:
    exat = 1 - d["is_Abnormal"].mean()
    print(f"{nome:10s} exatidão de 'sempre normal' = {exat:.1%} | anómalas detetadas = 0")

treino     exatidão de 'sempre normal' = 81.8% | anómalas detetadas = 0
validação  exatidão de 'sempre normal' = 78.9% | anómalas detetadas = 0


## 1.5 Função `avaliar()`

O código está em `avaliacao.py` (na raiz do repositório), para ser importado pelas secções seguintes e pelas Tasks 3 a 5 sem copiar código. A célula seguinte mostra o código no notebook.

Convenção: `score` é maior quando o risco é maior; há alerta se `score >= limiar`. A estabilidade por mês usa `avaliar_por_mes()`.

In [10]:
print(inspect.getsource(avaliacao))

"""Avaliação dos modelos — Task 2

Funções de métricas, folds temporais, baselines e limiar de decisão, usadas da Task 2 à Task 5.
Convenção: `score` = quanto maior, maior o risco de sessão anómala (classe 1).
"""
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, fbeta_score)


def limiar_por_orcamento(score, alertas_por_1000):
    """Limiar de score que gera ~`alertas_por_1000` alertas por 1000 sessões."""
    q = 1 - alertas_por_1000 / 1000
    return float(np.quantile(np.asarray(score, dtype=float), q))


def alertas_por_orcamento(score, alertas_por_1000, seed=0):
    """Alerta exatamente nas round(n·X/1000) sessões com score mais alto.

    Os empates no limite são resolvidos ao acaso. É necessário para scores com poucos
    valores distintos (ex.: regra binária), em que `score >= limiar` geraria muito mais
    ou muito menos alertas do que o orçamento.
    """
    score = np.asarray(score, dty

In [11]:
# Testes com respostas conhecidas
rng = np.random.default_rng(0)
y_t = (rng.random(20_000) < 0.2).astype(int)

# 1) Score perfeito
r = avaliar(y_t, y_t.astype(float), 0.5)
assert r["pr_auc"] == 1 and r["roc_auc"] == 1 and r["recall"] == 1 and r["precisao"] == 1

# 2) Score constante (equivale a "sempre normal"): PR-AUC = prevalência, ROC-AUC = 0,5, nenhum alerta
r = avaliar(y_t, np.zeros(len(y_t)), 1.0)
assert np.isclose(r["pr_auc"], y_t.mean()) and np.isclose(r["roc_auc"], 0.5)
assert r["recall"] == 0 and r["total_alertas_1000"] == 0

# 3) Alertar tudo: recall 1, 1000 alertas por 1000 sessões, precisão = prevalência
r = avaliar(y_t, rng.random(len(y_t)), 0.0)
assert r["recall"] == 1 and r["total_alertas_1000"] == 1000 and np.isclose(r["precisao"], y_t.mean())

# 4) Orçamento: o limiar gera o nº de alertas pedido
s = rng.random(len(y_t))
r = avaliar(y_t, s, limiar_por_orcamento(s, X_ALERTAS_1000))
assert abs(r["total_alertas_1000"] - X_ALERTAS_1000) < 1

# 5) Coerência: alertas corretos + falhadas = anómalas por 1000 sessões
assert np.isclose(r["alertas_corretos_1000"] + r["falhadas_1000"], 1000 * y_t.mean())

# 6) Score aleatório: PR-AUC ≈ prevalência, ROC-AUC ≈ 0,5
r = avaliar(y_t, rng.random(len(y_t)), limiar_por_orcamento(s, X_ALERTAS_1000))
assert abs(r["pr_auc"] - y_t.mean()) < 0.02 and abs(r["roc_auc"] - 0.5) < 0.02

print("Todos os testes passaram.")

Todos os testes passaram.


In [12]:
# Demonstração das métricas por mês (dados sintéticos: mostra o formato do resultado)
datas = pd.Series(pd.date_range("2021-01-01", periods=len(y_t), freq="h"))
score_demo = y_t * 0.5 + rng.random(len(y_t))
lim = limiar_por_orcamento(score_demo, X_ALERTAS_1000)
por_mes = avaliar_por_mes(datas, y_t, score_demo, lim)
print("ATENÇÃO: dados sintéticos, só para mostrar o formato. Não usar no relatório.")
por_mes[["n", "prevalencia", "pr_auc", "roc_auc", "recall", "total_alertas_1000"]].astype(float).round(3).head()

ATENÇÃO: dados sintéticos, só para mostrar o formato. Não usar no relatório.


,n,prevalencia,pr_auc,roc_auc,recall,total_alertas_1000
2021-01,744.0,0.198,0.729,0.872,0.599,211.022
2021-02,672.0,0.205,0.729,0.861,0.580,200.893
2021-03,744.0,0.237,0.723,0.851,0.540,193.548
2021-04,720.0,0.194,0.756,0.884,0.600,197.222
2021-05,744.0,0.199,0.786,0.893,0.689,217.742


# Task 2 (continuação) — Validação temporal, baselines e protocolo

## 2.1 Folds temporais

Expanding window com 4 folds de 3 meses. Em cada fold o treino contém todas as sessões
anteriores ao início da validação, e a validação são os 3 meses seguintes.
O teste (out–dez/2021) continua fora do notebook.

In [13]:
from avaliacao import folds_temporais, FOLDS

# Ordenar por tempo e usar índices posicionais (os folds devolvem posições)
dev = dev.sort_values("Start Time").reset_index(drop=True)

linhas = []
for nome, tr, va in folds_temporais(dev["Start Time"]):
    d_tr, d_va = dev.iloc[tr], dev.iloc[va]
    # Verificações: o treino acaba antes da validação, e nada do teste entra
    assert d_tr["Start Time"].max() < d_va["Start Time"].min()
    assert d_va["Start Time"].max() < VAL_END
    linhas.append({
        "fold": nome,
        "treino": f'{d_tr["Start Time"].min():%d/%m/%Y} – {d_tr["Start Time"].max():%d/%m/%Y}',
        "validação": f'{d_va["Start Time"].min():%d/%m/%Y} – {d_va["Start Time"].max():%d/%m/%Y}',
        "n treino": len(d_tr),
        "n validação": len(d_va),
        "prev. treino": round(d_tr["is_Abnormal"].mean(), 3),
        "prev. validação": round(d_va["is_Abnormal"].mean(), 3),
        "contas novas na validação": round((~d_va["UserID"].isin(set(d_tr["UserID"]))).mean(), 3),
    })

folds_resumo = pd.DataFrame(linhas).set_index("fold")
folds_resumo

,treino,validação,n treino,n validação,prev. treino,prev. validação,contas novas na validação
fold,,,,,,,
F1,31/12/2019 – 30/09/2020,01/10/2020 – 31/12/2020,125524,60364,0.196,0.168,0.345
F2,31/12/2019 – 31/12/2020,01/01/2021 – 31/03/2021,185888,41797,0.187,0.158,0.360
F3,31/12/2019 – 31/03/2021,01/04/2021 – 30/06/2021,227685,57542,0.182,0.182,0.459
F4,31/12/2019 – 30/06/2021,01/07/2021 – 30/09/2021,285227,75769,0.182,0.211,0.398


In [14]:
mensal = (dev.groupby(dev["Start Time"].dt.to_period("M"))["is_Abnormal"]
             .agg(sessoes="size", prevalencia="mean").round(3))
mensal["fold"] = ""
for nome, ini, fim in FOLDS:
    meses = pd.period_range(ini, pd.Timestamp(fim) - pd.Timedelta(days=1), freq="M")
    mensal.loc[mensal.index.isin(meses), "fold"] = nome
mensal.loc["2020-10":"2021-09"]

,sessoes,prevalencia,fold
Start Time,,,
2020-10,20252,0.170,F1
2020-11,19925,0.173,F1
2020-12,20187,0.163,F1
2021-01,16915,0.168,F2
2021-02,10283,0.171,F2
2021-03,14599,0.137,F2
2021-04,17307,0.156,F3
2021-05,20209,0.184,F3
2021-06,20026,0.203,F3


**Leitura dos folds**

- O primeiro fold começa em out/2020: o seu treino tem 9 meses (125 524 sessões) e já inclui 3 meses depois de a janela de 180 dias ficar completa (28/06/2020). Começar antes deixaria o treino dominado pelo período de arranque. Quatro folds de 3 meses cobrem um ano inteiro de validação, e cada validação tem pelo menos 41 797 sessões.
- As 4 validações cobrem um ano completo (out/2020 – set/2021), uma estação cada: outono, inverno, primavera e verão.
- O F2 (jan–mar/2021) é o fold mais pequeno (41 797 sessões): fevereiro tem apenas 10 283 sessões (Ano Novo Chinês),
  e janeiro e março também ficam abaixo dos restantes meses.
- A prevalência da validação varia entre 15,8 % (F2) e 21,1 % (F4) e não acompanha a do treino
  (nos F1 e F2 é inferior, no F4 é superior). A PR-AUC tem de ser lida face à prevalência de cada fold.
- Entre 34,5 % e 45,9 % das sessões de validação são de contas que não existem no treino do fold,
  o que confirma a exclusão de UserID em todos os folds e não só na partição da Task 1.
- O F4 coincide com a partição treino/validação da Task 1.

## 2.2 Pré-processamento dentro de cada fold

A preparação da Task 1 foi passada para uma Pipeline (`preparacao.py`). Em cada fold, a pipeline é ajustada
só com o treino desse fold e depois aplicada à validação, sem recalcular nada.

In [15]:
import preparacao
from preparacao import carregar_dados, criar_preprocessador, parametros_aprendidos, PREDITORES

dev = carregar_dados(DATA_PATH, ate=VAL_END)      # o teste continua de fora
print(dev.shape, "|", dev["Start Time"].min(), "→", dev["Start Time"].max())

# Os 33 preditores do preparacao.py são exatamente os da tabela da secção 1.2
assert set(PREDITORES) == set(preditores["variável"]), set(PREDITORES) ^ set(preditores["variável"])
# As variáveis tratadas como binárias são mesmo 0/1
assert all(set(dev[c].unique()) <= {0, 1} for c in preparacao.BIN)
print("Verificações OK")

(360996, 38) | 2019-12-31 21:27:03 → 2021-09-30 23:59:16
Verificações OK


In [16]:
param = {}
for nome, tr, va in folds_temporais(dev["Start Time"]):
    pre = criar_preprocessador(escalar=True).fit(dev.iloc[tr][PREDITORES])   # só o treino do fold
    Xva = pre.transform(dev.iloc[va][PREDITORES])                              # validação: só transform
    assert Xva.isna().sum().sum() == 0
    param[nome] = parametros_aprendidos(pre) | {"colunas finais": Xva.shape[1]}

pd.DataFrame(param).round(3)

,F1,F2,F3,F4
mediana post_hours_since_previous_completed_session,0.636,0.598,0.633,0.612
mediana post_hours_since_previous_completed_abnormal,19.044,19.441,20.427,20.166
mediana user_days_since_previous_completed_session,0.643,0.663,0.691,0.708
mediana user_days_since_previous_completed_abnormal,3.069,3.986,4.431,4.570
cap p99 user_active_sessions_at_start,5.000,4.000,4.000,3.000
média post_previous_sessions_7D,62.859,64.414,62.947,63.636
desvio post_previous_sessions_7D,36.145,33.883,32.985,32.289
média Temperature(℃),23.770,20.314,18.303,19.082
desvio Temperature(℃),6.774,8.272,8.807,8.261
categorias Charging Post ID,92.000,92.000,92.000,92.000


In [17]:
# O F4 tem o mesmo treino da Task 1 (dez/2019 – jun/2021), por isso tem de reproduzir
# os valores documentados no relatório da Task 1 (Tabela 23, secções 6.4 e 7.3).
f4 = param["F4"]
esperado = {
    "mediana post_hours_since_previous_completed_session": 0.612,
    "mediana post_hours_since_previous_completed_abnormal": 20.166,
    "mediana user_days_since_previous_completed_session": 0.708,
    "mediana user_days_since_previous_completed_abnormal": 4.570,
    "cap p99 user_active_sessions_at_start": 3,
    "colunas finais": 138,
}
for k, v in esperado.items():
    assert np.isclose(f4[k], v, atol=5e-4), (k, f4[k], v)
print("O F4 reproduz a preparação da Task 1.")

O F4 reproduz a preparação da Task 1.


**Leitura**

- Os valores aprendidos mudam de fold para fold: o cap passa de 5 (F1) para 3 (F4), a mediana dos dias desde a
  última anomalia do utilizador de 3,1 para 4,6 e a média da temperatura de 23,8 °C para 19,1 °C (o treino do F1
  tem sobretudo meses quentes). Usar os parâmetros da Task 1 em todos os folds seria fuga de informação, porque
  foram aprendidos com dados até jun/2021, que incluem as validações dos F1 a F3.
- O número de colunas é igual em todos os folds (138): os 92 postos existem desde o início, por isso o one-hot
  não muda.
- O F4 reproduz exatamente a preparação da Task 1.

## 2.3 Baselines

Todas as baselines são avaliadas nos 4 folds com a mesma função (`avaliar_folds`) que será usada para os modelos.
As que dependem de um orçamento alertam nas 200 sessões por 1000 com score mais alto de cada validação;
os empates são resolvidos ao acaso (semente fixa).

In [18]:
from avaliacao import avaliar_folds, resumir_folds
pd.set_option("display.max_columns", None)

rng_base = np.random.default_rng(42)

def taxa_posto_treino(tr, va):
    """Taxa de anómalas de cada posto, aprendida SÓ no treino do fold."""
    taxa = tr.groupby("Charging Post ID")["is_Abnormal"].mean()
    return va["Charging Post ID"].map(taxa).fillna(tr["is_Abnormal"].mean()).to_numpy()

BASELINES = {   # nome: (função de score, limiar fixo ou None = orçamento de 200)
    "Sempre normal": (lambda tr, va: np.zeros(len(va)), np.inf),
    "Aleatório": (lambda tr, va: rng_base.random(len(va)), None),
    "Taxa do posto (30 dias)": (lambda tr, va: va["post_previous_abnormal_rate_30D"].to_numpy(), None),
    "Taxa do posto (7 dias)": (lambda tr, va: va["post_previous_abnormal_rate_7D"].to_numpy(), None),
    "Última sessão do posto anómala (orçamento)": (lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), None),
    "Última sessão do posto anómala (regra)": (lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), 1),
    "Taxa do posto no treino do fold": (taxa_posto_treino, None),
}

res_base = {nome: avaliar_folds(dev, f, alertas_por_1000=X_ALERTAS_1000, limiar=lim)
            for nome, (f, lim) in BASELINES.items()}

# Verificações com respostas conhecidas
for nome, (f, lim) in BASELINES.items():
    if lim is None:   # orçamento: exatamente 200 alertas por 1000 sessões
        assert (res_base[nome]["total_alertas_1000"].sub(X_ALERTAS_1000).abs() < 0.5).all(), nome
sn, al = res_base["Sempre normal"], res_base["Aleatório"]
assert (sn["total_alertas_1000"] == 0).all() and np.allclose(sn["pr_auc"], sn["prevalencia"])
assert (al["pr_auc"].sub(al["prevalencia"]).abs() < 0.02).all() and (al["recall"].sub(0.2).abs() < 0.02).all()
print("Verificações das baselines OK")

Verificações das baselines OK


In [19]:
cols = ["prevalencia", "recall", "precisao", "pr_auc", "pr_auc_norm", "roc_auc", "f2",
        "alertas_corretos_1000", "falsos_alertas_1000", "falhadas_1000", "total_alertas_1000"]
tabela_base = pd.concat(res_base, names=["baseline", "fold"])[cols].round(3)
tabela_base

prevalencia  recall  precisao  pr_auc  pr_auc_norm  roc_auc     f2  alertas_corretos_1000  falsos_alertas_1000  falhadas_1000  total_alertas_1000
baseline                                   fold                                                                                                                                                   
Sempre normal                              F1          0.168   0.000     0.000   0.168        0.000    0.500  0.000                  0.000                0.000        168.428               0.000
                                           F2          0.158   0.000     0.000   0.158        0.000    0.500  0.000                  0.000                0.000        158.169               0.000
                                           F3          0.182   0.000     0.000   0.182        0.000    0.500  0.000                  0.000                0.000        182.249               0.000
                                           F4          0.211   0.000     0.000   0.211        0.000    0.500  0.000                  0.000                0.000        211.379               0.000
Aleatório                                  F1          0.168   0.202     0.170   0.172        0.004    0.508  0.194                 33.977              166.026        134.451             200.003
                                           F2          0.158   0.204     0.161   0.160        0.003    0.503  0.194                 32.275              167.715        125.894             199.990
                                           F3          0.182   0.200     0.182   0.183        0.001    0.504  0.196                 36.408              163.585        145.841             199.993
                                           F4          0.211   0.204     0.216   0.213        0.002    0.504  0.207                 43.223              156.779        168.156             200.003
Taxa do posto (30 dias)                    F1          0.168   0.464     0.391   0.429        0.313    0.754  0.447                 78.143              121.861         90.286             200.003
                                           F2          0.158   0.463     0.366   0.392        0.278    0.746  0.440                 73.283              126.708         84.886             199.990
                                           F3          0.182   0.487     0.444   0.437        0.311    0.772  0.478                 88.735              111.258         93.514             199.993
                                           F4          0.211   0.449     0.474   0.464        0.320    0.769  0.454                 94.880              105.122        116.499             200.003
Taxa do posto (7 dias)                     F1          0.168   0.492     0.414   0.499        0.398    0.764  0.474                 82.864              117.139         85.564             200.003
                                           F2          0.158   0.470     0.372   0.456        0.354    0.749  0.447                 74.383              125.607         83.786             199.990
                                           F3          0.182   0.525     0.478   0.547        0.446    0.789  0.515                 95.687              104.306         86.563             199.993
                                           F4          0.211   0.510     0.539   0.540        0.417    0.788  0.515                107.749               92.254        103.631             200.003
Última sessão do posto anómala (orçamento) F1          0.168   0.520     0.438   0.335        0.200    0.700  0.501                 87.519              112.484         80.909             200.003
                                           F2          0.158   0.482     0.381   0.294        0.161    0.677  0.458                 76.249              123.741         81.920             199.990
                                           F3          0.182   0.526     0.479   0.353        0.209    0.703  0.516                 95.878           

In [20]:
resumo_base = pd.DataFrame({n: resumir_folds(r) for n, r in res_base.items()}).T.round(3)
resumo_base

recall                  pr_auc                  pr_auc_norm                  roc_auc                  precisao                 
                                            média desvio pior fold  média desvio pior fold       média desvio pior fold   média desvio pior fold    média desvio pior fold
Sempre normal                               0.000  0.000     0.000  0.180  0.023     0.158       0.000  0.000     0.000   0.500  0.000     0.500    0.000  0.000     0.000
Aleatório                                   0.203  0.002     0.200  0.182  0.023     0.160       0.003  0.001     0.001   0.505  0.002     0.503    0.182  0.024     0.161
Taxa do posto (30 dias)                     0.466  0.016     0.449  0.430  0.030     0.392       0.306  0.019     0.278   0.760  0.013     0.746    0.419  0.049     0.366
Taxa do posto (7 dias)                      0.499  0.024     0.470  0.511  0.042     0.456       0.404  0.038     0.354   0.772  0.020     0.749    0.451  0.073     0.372
Última sessão do posto anómala (orçamento)  0.511  0.020     0.482  0.344  0.042     0.294       0.201  0.030     0.161   0.698  0.015     0.677    0.461  0.070     0.381
Última sessão do posto anómala (regra)      0.504  0.038     0.455  0.344  0.042     0.294       0.201  0.030     0.161   0.698  0.015     0.677    0.504  0.038     0.455
Taxa do posto no treino do fold             0.371  0.026     0.345  0.321  0.037     0.290       0.172  0.026     0.146   0.709  0.015     0.687    0.333  0.044     0.290

In [21]:
# Referências que os modelos têm de superar, fold a fold
REF_RECALL = res_base["Última sessão do posto anómala (orçamento)"]["recall"]   # métrica principal
REF_PR_AUC_NORM = res_base["Taxa do posto (7 dias)"]["pr_auc_norm"]            # melhor ordenação
pd.DataFrame({"recall@200 a superar": REF_RECALL, "PR-AUC normalizada a superar": REF_PR_AUC_NORM}).round(3)

,recall@200 a superar,PR-AUC normalizada a superar
F1,0.520,0.398
F2,0.482,0.354
F3,0.526,0.446
F4,0.518,0.417


**Leitura das baselines**

- As baselines sem capacidade confirmam os valores esperados: PR-AUC igual à prevalência, ROC-AUC de 0,5
  e, no caso da aleatória, recall de ~0,20 com 200 alertas por 1000 sessões.
- A taxa de anomalias do posto nos 7 dias anteriores é a melhor baseline em ordenação
  (PR-AUC média de 0,511 e PR-AUC normalizada de 0,404; ROC-AUC de 0,772). É melhor do que a de 30 dias (0,430).
- No ponto de decisão (200 alertas por 1000), a regra "a última sessão do posto foi anómala" tem o melhor
  recall (0,511) e o melhor pior fold (0,482), apesar de ter PR-AUC mais baixa (0,344):
  a ordem das baselines muda conforme a métrica.
- A taxa fixa de cada posto, aprendida no treino do fold, é bastante pior (PR-AUC de 0,321):
  o que prevê as falhas é o histórico recente do posto, não o posto em si.
- O F2 é o pior fold em quase todas as baselines e métricas; a exceção é o recall da taxa a 30 dias, cujo pior fold é o F4 (0,449).
- Referências para os modelos: PR-AUC normalizada da taxa de 7 dias e recall@200 da regra da última sessão, fold a fold.

## 2.4 Limiar: transferência entre períodos

No teste, o limiar tem de ser fixado antes, com dados anteriores. Aqui simula-se esse procedimento nos folds:
o limiar que gera 200 alertas por 1000 sessões num período é aplicado ao período seguinte.

In [22]:
from avaliacao import agregar_folds, comparar, transferir_limiar

COLS_BASE = {"Taxa do posto (7 dias)": "post_previous_abnormal_rate_7D",
             "Taxa do posto (30 dias)": "post_previous_abnormal_rate_30D",
             "Última sessão do posto anómala (orçamento)": "post_last_completed_abnormal"}
sc_base = {}
for nome, col in COLS_BASE.items():
    r, sc_base[nome] = avaliar_folds(dev, lambda tr, va, c=col: va[c].to_numpy(),
                                     alertas_por_1000=X_ALERTAS_1000, devolver_scores=True)
    assert r.equals(res_base[nome])          # mesmos resultados da 2.3

# Transferência do limiar (só scores contínuos; a regra binária não tem quantil útil)
pd.concat({n: transferir_limiar(sc_base[n], X_ALERTAS_1000)
           for n in ["Taxa do posto (7 dias)", "Taxa do posto (30 dias)"]}).round(3)

limiar  alertas por 1000 (limiar transferido)  recall (limiar transferido)  precisão (limiar transferido)  recall (orçamento exato)
Taxa do posto (7 dias)  F1 → F2   0.243                                186.353                        0.453                          0.385                     0.470
                        F2 → F3   0.235                                221.004                        0.554                          0.456                     0.525
                        F3 → F4   0.250                                271.707                        0.608                          0.473                     0.510
Taxa do posto (30 dias) F1 → F2   0.251                                197.263                        0.460                          0.369                     0.463
                        F2 → F3   0.249                                205.015                        0.495                          0.440                     0.487
                        F3 → F4   0.253                                287.783                        0.594                          0.436                     0.449

In [23]:
from avaliacao import avaliar_limiar_movel

# Limiar recalculado todas as segundas-feiras com os scores das sessões das últimas 7 ou 28 dias
mov = {}
for nome in ["Taxa do posto (7 dias)", "Taxa do posto (30 dias)"]:
    for janela in (7, 28):
        mov[(nome, f"{janela} dias")] = avaliar_limiar_movel(dev, dev[COLS_BASE[nome]],
                                                             X_ALERTAS_1000, janela_dias=janela)
pd.concat(mov, names=["baseline", "janela", "fold"]).round(3)

alertas por 1000 (limiar móvel)  alertas por 1000 — semana mín.  alertas por 1000 — semana máx.  recall (limiar móvel)  precisão (limiar móvel)  \
baseline                janela  fold                                                                                                                                                    
Taxa do posto (7 dias)  7 dias  F1                            213.273                         153.949                         272.157                  0.510                    0.403   
                                F2                            191.401                         110.915                         283.790                  0.460                    0.380   
                                F3                            208.300                         163.286                         284.884                  0.535                    0.468   
                                F4                            213.293                         154.017                         314.663                  0.523                    0.519   
                        28 dias F1                            200.898                         131.769                         245.074                  0.493                    0.413   
                                F2                            188.267                         143.795                         272.336                  0.456                    0.383   
                                F3                            216.329                         168.771                         291.990                  0.544                    0.458   
                                F4                            213.491                         146.976                         286.913                  0.527                    0.522   
Taxa do posto (30 dias) 7 dias  F1                            204.493                         147.315                         243.019                  0.472                    0.389   
                                F2                            186.903                         139.199                         255.501                  0.442                    0.374   
                                F3                            205.658                         133.545                         244.401                  0.488                    0.433   
                                F4                            207.552                         174.878                         264.297                  0.473                    0.481   
                        28 dias F1                            204.890                         156.867                         234.606                  0.475                    0.391   
                                F2                            188.554                         119.501                         259.914                  0.444                    0.373   
                                F3                            216.937                         169.273                         270.457                  0.510                    0.428   
                                F4                            219.391                         165.217                         305.653                  0.486                    0.468   

                                      recall (orçamento exato)  alertas por 1000 (quantil do próprio período)  
baseline                janela  fold                                                                           
Taxa do posto (7 dias)  7 dias  F1                       0.492                                        200.583  
                                F2                       0.470                                        200.971  
                                F3                       0.525                                        205.294  
                                F4                       0.510                                        200.095  
                        28 dias F1 

**Leitura**

- **Limiar fixo** (copiado do trimestre anterior): 186 a 288 alertas por 1000 sessões em vez de 200. Quando as
  falhas aumentam (jul–set), o mesmo limiar gera até 44 % mais alertas. **Rejeitado.**
- **Limiar móvel** (recalculado à segunda-feira com os scores dos dias anteriores): 187 a 213 alertas por 1000 sessões por fold com janela de 7 dias, e 188 a 219 com janela de 28 dias.
- Na configuração escolhida (taxa a 7 dias, janela de 7 dias), o recall com limiar móvel fica a −0,010 a +0,018 do recall com orçamento exato; em todas as configurações testadas, a diferença fica entre −0,021 e +0,037. O recall@200 dos folds é, por isso, uma boa estimativa do recall em operação.
- A carga semanal varia (111 a 315 alertas por 1000): o orçamento é cumprido por trimestre, não em cada semana.
- **[DECISÃO] Janela de 7 dias.** Equivalente à de 28 por fold, adapta-se mais depressa e, no teste, só a primeira
  semana usa scores de sessões de treino do modelo final.

## 2.5 Agregação dos resultados dos folds

In [24]:
agreg = pd.concat({n: agregar_folds(res_base[n], sc_base[n]) for n in COLS_BASE})
agreg.round(3)

média  desvio  pior fold fold do pior  média ponderada  tendência por fold  pooled
Taxa do posto (7 dias)                     recall       0.499   0.024      0.470           F2            0.502               0.011   0.503
                                           pr_auc       0.511   0.042      0.456           F2            0.516               0.021   0.524
                                           pr_auc_norm  0.404   0.038      0.354           F2            0.408               0.015   0.417
                                           roc_auc      0.772   0.020      0.749           F2            0.775               0.011   0.778
                                           precisao     0.451   0.073      0.372           F2            0.462               0.048   0.463
Taxa do posto (30 dias)                    recall       0.466   0.016      0.449           F4            0.465              -0.002   0.464
                                           pr_auc       0.430   0.030      0.392           F2            0.436               0.015   0.442
                                           pr_auc_norm  0.306   0.019      0.278           F2            0.309               0.006   0.316
                                           roc_auc      0.760   0.013      0.746           F2            0.762               0.007   0.765
                                           precisao     0.419   0.049      0.366           F2            0.426               0.033   0.426
Última sessão do posto anómala (orçamento) recall       0.511   0.020      0.482           F2            0.514               0.004   0.515
                                           pr_auc       0.344   0.042      0.294           F2            0.351               0.024   0.354
                                           pr_auc_norm  0.201   0.030      0.161           F2            0.206               0.014   0.208
                                           roc_auc      0.698   0.015      0.677           F2            0.701               0.006   0.702
                                           precisao     0.461   0.070      0.381           F2            0.473               0.043   0.473

In [25]:
pd.DataFrame({
    "7 dias vs 30 dias (PR-AUC)": comparar(res_base["Taxa do posto (7 dias)"],
                                           res_base["Taxa do posto (30 dias)"], "pr_auc"),
    "Última sessão vs 7 dias (recall@200)": comparar(res_base["Última sessão do posto anómala (orçamento)"],
                                                     res_base["Taxa do posto (7 dias)"], "recall"),
}).round(3)

,7 dias vs 30 dias (PR-AUC),Última sessão vs 7 dias (recall@200)
diferença média,0.080,0.012
desvio da diferença,0.020,0.011
folds em que A ganha,4.000,4.000
pior fold A,0.456,0.482
pior fold B,0.392,0.470


**Leitura**

- Média simples, média ponderada e pooled diferem no máximo 0,013: a escolha da agregação não muda as conclusões.
- Tendência da taxa a 7 dias: PR-AUC normalizada +0,015 por fold e ROC-AUC +0,011, ou seja, sem deterioração.
- Comparação emparelhada: a regra da última sessão supera a taxa a 7 dias no recall@200 em 4 de 4 folds
  (+0,012, com melhor pior fold). A taxa a 7 dias supera a de 30 dias na PR-AUC em todos os folds.

## 2.6 Protocolo de avaliação

| Elemento | Definição |
|---|---|
| Preditores | 33 (30 numéricos, 3 categóricos) em 6 grupos; tabela da secção 1.2 |
| Validação | 4 folds em expanding window; validações trimestrais de out/2020 a set/2021 |
| Teste | out–dez/2021, excluído na leitura dos dados; usado uma única vez no fim |
| Pré-processamento | Pipeline (`preparacao.py`) reajustada em cada fold só com o treino do fold |
| Métrica principal | Recall com 200 alertas por 1000 sessões |
| Complementares | PR-AUC normalizada, PR-AUC, ROC-AUC, precisão, F2 |
| Operacionais | Alertas corretos, falsos alertas, falhas não detetadas e total por 1000; alertas por semana |
| Estabilidade | Desvio-padrão, pior fold, tendência, métricas por mês |
| Limiar | Orçamento X = 200; limiar móvel semanal (quantil 80 % dos scores dos 7 dias anteriores) |
| Baselines | Sempre normal, aleatório, taxa do posto 7 e 30 dias, última sessão anómala, taxa fixa do posto |
| Referências a superar | Recall@200 da última sessão anómala e PR-AUC normalizada da taxa a 7 dias, em todos os folds |
| Agregação | Média simples; desvio e pior fold; ponderada e pooled como confirmação |
| Regra de decisão | A > B se: maior recall médio, ganha em ≥ 3 de 4 folds e pior fold não inferior |
| Hiperparâmetros | Escolhidos nos mesmos 4 folds, pelo recall@200 médio e pela regra de decisão acima; o teste não é usado |
| Modelo final | Retreinado com todo o desenvolvimento (dez/2019 – set/2021) |

In [26]:
print(f"tempo total deste caderno: {time.perf_counter() - T_INICIO:.0f} s")

tempo total deste caderno: 15 s
